# 26 — Online loop detection and RPMR timing in Colab

Notebook 25 passed. **Do not rerun it.** Upload this notebook into a fresh Colab GPU
runtime and choose **Run all**. Allow Drive and enter the current unit balance and live rate.
No Git clone, training, adapters, or local tools are needed.

**Scope:** test online instrumentation and measure its overhead. Two hand-written code
questions, three methods, one deterministic try each, at most **384 generated tokens**.
No model-written code is executed. No code accuracy or research improvement is claimed.

Methods: ordinary thinking; the same decoding with detection/logging only; detection plus RPMR.
The detector looks for **three consecutive identical token blocks** of length 8, 16, or 32,
inside thinking. It detects exact text cycles, not every semantic loop or wasted thought.
Repair is never forced. If no loop appears, the result must say recovery was not tested.

**Question:** can the detector and memory edit run online, and what do they cost?
**Hypothesis:** logging alone leaves greedy output unchanged; triggered repair preserves selected readouts.
**Change:** detection and conditional memory edit. **Measure:** triggers, edits, output tokens,
decode time, peak allocated memory, and immediate readout error.
**Reference:** ordinary thinking and detection-only on the same prompt.
**Data:** two development questions below, not the existing 464 benchmark problems.
**Support:** outputs match without edits and online checks pass. **Contradiction:** logging
changes output, wrong cache shapes, invalid edits, or budget exhaustion.

**Budget:** keep 20 units; this session uses at most **three estimated units**.
At the recorded A100 rate of 6.77/hour, that allows about **26.6 minutes**.
Planning estimate **4–10 minutes**, roughly **0.45–1.13 units** at that historical rate.
This assumes similar setup and short-prefix speed to Notebook 25; long-prefix and repair
costs remain unknown. Downloads, rates, GPU choice and cache behavior can change the time.
The three-unit ceiling includes setup after cell 1 starts. No larger pilot follows automatically.

Six responses × 384 tokens = at most 2,304 new tokens. Directions come from the observed
repeated span's memory drift; the prompt-derived protected basis is fixed within a response.
At most two edits per response, at least 64 tokens apart, with at most 5% memory-norm change.

Saves `report.json` and every response/event in `answers.jsonl` to:
`MyDrive/stop-overthinking/rpmr-online-smoke/<unique-run>/`.
It attempts runtime disconnection on success/failure; a wall-time watchdog also attempts it
near the estimated cap. The entered rate is an estimate, not live account billing.
If it cannot disconnect, use **Runtime → Disconnect and delete runtime**.
**Prepared, not run in Colab yet.**


## 1. Budget and Drive

Counts setup time from this cell's start, saves progress, and starts the same watchdog
as Notebook 25. Enter the actual current balance; do not assume exactly 50 remains.


In [ ]:
import sys, time, json, math, threading, subprocess, datetime, uuid
from pathlib import Path
from google.colab import drive, runtime

SESSION_START = time.monotonic()
SMOKE_CAP = 3.0
RESERVE = 20.0
BALANCE = float(input("Current Colab compute units available: "))
RATE = float(input("LIVE usage rate in compute units per hour, from Colab Resources: "))
if not math.isfinite(BALANCE) or not math.isfinite(RATE) or RATE <= 0 or BALANCE <= RESERVE:
    raise ValueError("Enter a finite positive rate and balance above the 20-unit reserve.")
SMOKE_CAP = min(SMOKE_CAP, BALANCE - RESERVE)
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = Path('/content/drive/MyDrive/stop-overthinking/rpmr-online-smoke') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
REPORT = dict(status='started', run_id=RUN_ID, model='Qwen/Qwen3.5-2B',
              initial_balance=BALANCE, entered_units_per_hour=RATE,
              estimated_unit_cap=SMOKE_CAP, reserve_units=RESERVE, checks=[],
              scope='online instrumentation and timing; no benchmark accuracy')
LOCK = threading.RLock()
STOP_WATCHDOG = threading.Event()

def used_units():
    return (time.monotonic() - SESSION_START) / 3600 * RATE

def save_report():
    with LOCK:
        REPORT['wall_seconds'] = round(time.monotonic() - SESSION_START, 3)
        REPORT['estimated_units_used'] = round(used_units(), 4)
        temporary = OUT / 'report.json.tmp'
        temporary.write_text(json.dumps(REPORT, indent=2))
        temporary.replace(OUT / 'report.json')

def guard():
    if used_units() >= max(0, SMOKE_CAP - 0.1):
        raise TimeoutError('Estimated smoke-unit ceiling reached. Stop this session.')

def watchdog():
    while not STOP_WATCHDOG.wait(1):
        if used_units() >= max(0, SMOKE_CAP - 0.1):
            with LOCK:
                REPORT['status'] = 'budget_stopped'
                save_report()
            print('Budget ceiling: disconnecting runtime. Report saved to Drive.', flush=True)
            try:
                runtime.unassign()
            except Exception as exc:
                with LOCK:
                    REPORT['disconnect_error'] = str(exc)
                    save_report()
                print('Disconnect manually: Runtime → Disconnect and delete runtime.', flush=True)
            return

save_report()
threading.Thread(target=watchdog, daemon=True).start()
print(f'Smoke cap {SMOKE_CAP:.2f} units; reserve {RESERVE:.0f}. Results: {OUT}')
guard()


## 2. Calculation and exact-repeat detector

Reuses the protected calculation from Notebook 25. The detector runs only on the
already-generated prefix. A synthetic repeated sequence checks the detector here in Colab;
it is not counted as a natural model loop.


In [ ]:
def orthogonal_basis(torch, columns, reference_scale=None, max_rank=None):
    if columns.shape[1] == 0:
        return columns[:, :0]
    u, singular, _ = torch.linalg.svd(columns, full_matrices=False)
    scale = singular[0] if reference_scale is None else reference_scale
    threshold = max(columns.shape) * torch.finfo(columns.dtype).eps * scale
    rank = int((singular > threshold).sum().item())
    if max_rank is not None:
        rank = min(rank, max_rank)
    return u[:, :rank]

def repair(torch, state, reference, probes, cycle, eta=0.25, cap_fraction=0.05):
    if state.ndim != 2 or state.shape != reference.shape:
        raise ValueError('Current and reference memory must have the same matrix shape.')
    if probes.ndim != 2 or cycle.ndim != 2 or probes.shape[0] != state.shape[0] or cycle.shape[0] != state.shape[0]:
        raise ValueError('Direction rows must match the memory key dimension.')
    if not 0 <= eta <= 1 or not 0 < cap_fraction <= 1:
        raise ValueError('Invalid repair strength or norm cap.')
    s, ref, q, c = [x.detach().float() for x in (state, reference, probes, cycle)]
    if not all(torch.isfinite(x).all().item() for x in (s, ref, q, c)):
        raise ValueError('Non-finite memory or direction.')
    u = orthogonal_basis(torch, q)
    scale = torch.linalg.matrix_norm(c, ord=2) if c.shape[1] else 0.0
    v = orthogonal_basis(torch, c - u @ (u.T @ c), reference_scale=scale)
    drift = v @ (v.T @ (s - ref))
    proposed_norm = eta * torch.linalg.norm(drift).item()
    allowed_norm = cap_fraction * torch.linalg.norm(s).item()
    effective_eta = eta * min(1.0, allowed_norm / max(proposed_norm, 1e-30))
    edited = s - effective_eta * drift
    error = torch.linalg.norm(u.T @ (edited - s)).item()
    return edited, u, v, dict(protected_rank=u.shape[1], eligible_rank=v.shape[1],
                              effective_eta=effective_eta, immediate_readout_error=error,
                              edit_norm=torch.linalg.norm(edited-s).item())


def repeated_period(tokens):
    for period in (8, 16, 32):
        if len(tokens) >= 3 * period:
            if tokens[-period:] == tokens[-2*period:-period] == tokens[-3*period:-2*period]:
                return period
    return None

assert repeated_period(list(range(8)) * 3) == 8
assert repeated_period(list(range(16)) * 3) == 16
assert repeated_period(list(range(32)) * 3) == 32
assert repeated_period(list(range(100))) is None
assert repeated_period(list(range(8)) * 2) is None
assert repeated_period(list(range(8)) * 3 + [1000]) is None
REPORT['checks'].append('Synthetic detector cases passed in this Colab kernel')
save_report()
MAX_TOKENS = 384
MAX_EDITS = 2
COOLDOWN = 64
QUESTION_LIST = [
    'Write a Python function that returns the sum of even integers in a list. Briefly plan your approach.',
    'Write a Python function longest_unique_substring_length(text) that returns the length of the longest contiguous substring with no repeated characters. Explain how to handle empty text and repeated characters.'
]
print('Two prompts, three methods each, 384 TOTAL new tokens per response. No training.')


## 3. Run, save each answer, report overhead, and disconnect

The detection-only response must match ordinary greedy decoding. Repair occurs only
when repeated thinking is detected and an eligible memory direction exists.
Every trigger, skipped edit, repair, raw output and elapsed time is saved.
Cached memory is edited after the current forward call; its effect starts with later
forward calls. Pending next-token scores are not retroactively recomputed.
Short 384-token caps can miss loops appearing later. Passing instrumentation is not
evidence of better accuracy, and no-loop results are explicitly labelled.


In [ ]:
try:
    guard()
    remaining_seconds = max(1, (SMOKE_CAP-used_units()-0.1)/RATE*3600)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==5.18.0', 'accelerate', 'safetensors'],
                   check=True, timeout=remaining_seconds)
    guard()
    import os, inspect, hashlib
    from collections import deque
    import torch, transformers
    from transformers import AutoTokenizer
    from transformers.models.qwen3_5 import modeling_qwen3_5 as implementation
    if not torch.cuda.is_available():
        raise RuntimeError('Choose a fresh Colab GPU runtime first.')
    torch.manual_seed(3407)
    torch.cuda.manual_seed_all(3407)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    MODEL_ID = 'Qwen/Qwen3.5-2B'
    REVISION = '15852e8c16360a2fea060d615a32b45270f8a8fc'
    REPORT['model_revision'] = REVISION
    REPORT['environment'] = dict(python=sys.version, torch=torch.__version__,
                                 transformers=transformers.__version__,
                                 gpu=torch.cuda.get_device_name(0), model_dtype=str(dtype),
                                 memory_math_dtype='torch.float32', seed=3407)
    REPORT['settings'] = dict(max_total_new_tokens=MAX_TOKENS, max_edits=MAX_EDITS,
                              cooldown_tokens=COOLDOWN, periods=[8,16,32], repetitions=3,
                              decoding='greedy', optional_acceleration_packages_installed=False)
    save_report()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
    loader = getattr(transformers, 'AutoModelForMultimodalLM', None)
    if loader is None:
        loader = transformers.AutoModelForImageTextToText
    model = loader.from_pretrained(MODEL_ID, revision=REVISION, dtype=dtype,
                                    device_map={'': 0}, attn_implementation='sdpa')
    model.eval()
    selected = next(m for m in model.modules() if m.__class__.__name__ == 'Qwen3_5GatedDeltaNet')
    layer_index = selected.layer_idx
    REPORT['selected_layer'] = layer_index
    REPORT['selected_head'] = 0
    REPORT['implementation_sha256'] = hashlib.sha256(Path(inspect.getfile(implementation)).read_bytes()).hexdigest()
    eos = model.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos])
    saved = []
    ANSWERS = OUT / 'answers.jsonl'

    def prefill(question):
        guard()
        prompt = tokenizer.apply_chat_template([dict(role='user',content=question)],
                                               tokenize=False, add_generation_prompt=True, enable_thinking=True)
        ids = tokenizer(prompt,return_tensors='pt').input_ids.to('cuda')
        captured = dict(active=False, probes=None)
        def start_capture(module,args):
            captured['active'] = True
        def end_capture(module,args,output):
            captured['active'] = False
        handles = [selected.register_forward_pre_hook(start_capture),selected.register_forward_hook(end_capture)]
        originals = {}
        def wrap_rule(original):
            def wrapped(*args,**kwargs):
                if captured['active'] and captured['probes'] is None:
                    query = args[0] if args else kwargs['query']
                    vectors = torch.nn.functional.normalize(query[0,:,0,:].detach().float(),dim=-1)
                    captured['probes'] = orthogonal_basis(torch,vectors.T,max_rank=8)
                return original(*args,**kwargs)
            return wrapped
        for name in ('torch_chunk_gated_delta_rule','torch_recurrent_gated_delta_rule'):
            originals[name] = getattr(implementation,name)
            setattr(implementation,name,wrap_rule(originals[name]))
        try:
            with torch.no_grad():
                result = model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=True,logits_to_keep=1)
        finally:
            for handle in handles:
                handle.remove()
            for name,original in originals.items():
                setattr(implementation,name,original)
        if captured['probes'] is None:
            raise RuntimeError('Active kernel bypassed query capture.')
        memory = result.past_key_values.layers[layer_index].recurrent_states[0]
        if memory.dtype != torch.float32 or tuple(memory.shape[-2:]) != (128,128):
            raise RuntimeError('Cache shape or precision changed from Notebook 25.')
        return ids, result, captured['probes']

    def run_answer(prompt_index, question, method):
        guard()
        torch.cuda.synchronize()
        overall_start = time.monotonic()
        ids, output, probes = prefill(question)
        cache = output.past_key_values
        cursor = output.logits[:,-1].argmax(dim=-1,keepdim=True)
        tokens, events, edits = [], [], 0
        last_episode = -10000
        history = deque(maxlen=129)
        if method != 'ordinary':
            history.append((0,cache.layers[layer_index].recurrent_states[0][0,0].detach().clone()))
        torch.cuda.synchronize()
        decode_start = time.monotonic()
        in_thinking = True
        with torch.no_grad():
            for step in range(MAX_TOKENS):
                guard()
                token = int(cursor.item())
                tokens.append(token)
                if token in eos_ids:
                    break
                output = model(input_ids=cursor,
                    attention_mask=torch.ones((1,ids.shape[1]+len(tokens)),device='cuda',dtype=torch.long),
                    past_key_values=cache,use_cache=True,logits_to_keep=1)
                if not torch.isfinite(output.logits).all().item():
                    raise RuntimeError('Non-finite logits.')
                cache = output.past_key_values
                cursor = output.logits[:,-1].argmax(dim=-1,keepdim=True)
                if method != 'ordinary':
                    state = cache.layers[layer_index].recurrent_states[0][0,0]
                    history.append((len(tokens),state.detach().clone()))
                    # The prompt supplies the opening think tag; stop edits after closure.
                    if '</think>' in tokenizer.decode(tokens,skip_special_tokens=False):
                        in_thinking = False
                    period = repeated_period(tokens) if in_thinking else None
                    if period is not None and len(tokens)-last_episode >= COOLDOWN:
                        last_episode = len(tokens)
                        event = dict(at_token=len(tokens),period=period,repeats=3,thinking=True,
                                     action='detected_only',edit_info=None)
                        if method == 'rpmr' and edits < MAX_EDITS and len(tokens) < MAX_TOKENS:
                            reference_index = len(tokens)-3*period
                            reference = next((m for n,m in history if n==reference_index),None)
                            if reference is None:
                                event['action'] = 'skipped_missing_reference'
                            else:
                                cycle = orthogonal_basis(torch,state.detach().float()-reference,max_rank=4)
                                edited,u,v,info = repair(torch,state,reference,probes,cycle)
                                torch.testing.assert_close(u.T@edited,u.T@state,atol=3e-5,rtol=3e-5)
                                if info['edit_norm'] > 0 and info['eligible_rank'] > 0:
                                    state.copy_(edited)
                                    history[-1] = (len(tokens),state.detach().clone())
                                    edits += 1
                                    event['action'] = 'repaired'
                                else:
                                    event['action'] = 'skipped_no_eligible_change'
                                event['edit_info'] = info
                        elif method == 'rpmr':
                            event['action'] = 'skipped_edit_limit_or_end_of_response'
                        events.append(event)
                        # Persist an event immediately; the complete answer is saved below.
                        with LOCK:
                            REPORT['last_event'] = dict(prompt_index=prompt_index,method=method,**event)
                            save_report()
        torch.cuda.synchronize()
        decode_seconds = time.monotonic()-decode_start
        answer = dict(prompt_index=prompt_index,question=question,method=method,
                      generated_tokens=len(tokens),token_ids=tokens,
                      raw_output=tokenizer.decode(tokens,skip_special_tokens=False),
                      decode_seconds=decode_seconds,tokens_per_second=len(tokens)/max(decode_seconds,1e-9),
                      response_wall_seconds=time.monotonic()-overall_start,
                      hit_total_token_cap=len(tokens)==MAX_TOKENS and tokens[-1] not in eos_ids,
                      thinking_closed='</think>' in tokenizer.decode(tokens,skip_special_tokens=False),
                      detected_episodes=len(events),repairs=edits,events=events,
                      peak_allocated_gpu_memory_gb=torch.cuda.max_memory_allocated()/1e9)
        with ANSWERS.open('a') as stream:
            stream.write(json.dumps(answer)+'\n')
            stream.flush()
            os.fsync(stream.fileno())
        saved.append(answer)
        with LOCK:
            REPORT['completed_answers'] = len(saved)
            REPORT['summary'] = [{k:r[k] for k in ('prompt_index','method','generated_tokens',
                'decode_seconds','tokens_per_second','detected_episodes','repairs','hit_total_token_cap')} for r in saved]
            save_report()
        print(f'Prompt {prompt_index+1} | {method} | {len(tokens)} tokens | {decode_seconds:.1f}s | {len(events)} episodes | {edits} repairs',flush=True)
        del cache, output, history
        torch.cuda.empty_cache()
        return answer

    # Reverse order on the second prompt to reduce simple first-run timing bias.
    for index, question in enumerate(QUESTION_LIST):
        order = ['ordinary','detect_only','rpmr'] if index==0 else ['rpmr','detect_only','ordinary']
        results = {method:run_answer(index,question,method) for method in order}
        if results['ordinary']['token_ids'] != results['detect_only']['token_ids']:
            raise RuntimeError('Detection-only changed greedy output; comparison is invalid.')
        if results['rpmr']['repairs']==0 and results['ordinary']['token_ids'] != results['rpmr']['token_ids']:
            raise RuntimeError('RPMR changed output without an edit; comparison is invalid.')
        REPORT['checks'].append(f'Prompt {index+1}: detection-only output exactly matches ordinary output')
        ratio = results['detect_only']['decode_seconds']/max(results['ordinary']['decode_seconds'],1e-9)
        REPORT.setdefault('rough_instrumentation_overhead',[]).append(dict(prompt_index=index,
            detect_only_vs_ordinary_wall_ratio=ratio,
            note='Same greedy output; very small timing sample, not a throughput guarantee.'))
        save_report()
    total_edits = sum(r['repairs'] for r in saved)
    total_episodes = sum(r['detected_episodes'] for r in saved)
    REPORT['natural_detected_episodes'] = total_episodes
    REPORT['actual_online_repairs'] = total_edits
    REPORT['status'] = 'online_checks_passed' if total_edits else 'instrumentation_passed_no_online_repairs'
    REPORT['interpretation'] = ('Online plumbing exercised; assess saved text before claiming loop escape.' if total_edits
        else 'No eligible natural repair occurred. Timing/instrumentation checked; loop recovery remains untested.')
    REPORT['accuracy_measured'] = False
    REPORT['next_step'] = 'Review saved events, text and complete timing before expanding the workload.'
    guard()
except Exception as exc:
    with LOCK:
        if REPORT.get('status') != 'budget_stopped':
            REPORT['status'] = 'failed'
        REPORT['error'] = f'{type(exc).__name__}: {exc}'
    import traceback
    traceback.print_exc()
finally:
    save_report()
    print(json.dumps(REPORT,indent=2))
    print('Drive report:', OUT/'report.json')
    try:
        from google.colab import files
        import zipfile
        archive = Path('/content') / f'rpmr-online-{RUN_ID}.zip'
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zipped:
            for path in (OUT/'report.json',OUT/'answers.jsonl'):
                if path.exists():
                    zipped.write(path,path.name)
        files.download(str(archive))
    except Exception as exc:
        print('Download unavailable; saved files remain on Drive:',exc)
    try:
        runtime.unassign()
    except Exception as exc:
        REPORT['disconnect_error'] = str(exc)
        save_report()
        print('Disconnect manually: Runtime → Disconnect and delete runtime.')
    else:
        STOP_WATCHDOG.set()
